# Fine-Tuning in Action — Exercise + Solution Notebook

Train GPT-2 on **The Everglades Cipher** corpus using six fine-tuning techniques.

Each section has:

1. An **exercise cell** with `# TODO` stubs — complete it yourself first.
2. A collapsed **solution cell** — click  to reveal the reference implementation.

## Techniques

| #   | Technique             | Description                                     |
| --- | --------------------- | ----------------------------------------------- |
| 1   | Continued Pretraining | Extend GPT-2's knowledge on raw domain text     |
| 2   | SFT                   | Instruction-following on Q&A pairs              |
| 3   | DPO                   | Preference alignment with chosen/rejected pairs |
| 4   | Full Fine-Tuning      | Update all 117M parameters                      |
| 5   | LoRA                  | Parameter-efficient adapters (~0.5% of params)  |
| 6   | QLoRA                 | 4-bit quantized base + LoRA adapters            |

> **Note**: All paths assume the Jupyter kernel CWD is the repo root (`c:\r\ai-portfolio`).


## Checkpoint Dependencies

![Combined notebook checkpoint map showing data preparation, pretraining, SFT, DPO, LoRA, and evaluation dependencies](images/combined-notebook-checkpoints.png)

This map tracks the artifacts and checkpoints used across the combined notebook. It is an execution and dependency guide, not a universal claim that LoRA must follow DPO.


## Section 0: Setup


In [ ]:
# Install required packages (run once if needed)
# %pip install transformers peft trl datasets torch accelerate bitsandbytes -q

In [ ]:
import os, json, glob, textwrap
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from transformers import DataCollatorForLanguageModeling
from datasets import Dataset
import warnings

warnings.filterwarnings("ignore")

# Kernel CWD is the repo root (c:\r\ai-portfolio).
ROOT = Path("learning/genai/04-llm")
DATA_DIR = ROOT / "data"
CONTENT_DIR = ROOT / "content" / "the-everglades-cipher"
CKPT_DIR = ROOT / "checkpoints"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"ROOT resolved to: {ROOT.resolve()}")
print(f"Content dir exists: {CONTENT_DIR.exists()}")
print(f"Data dir exists: {DATA_DIR.exists()}")

## Section 1: Load Data


In [ ]:
def load_jsonl(path):
    # TODO: implement this function to load a JSONL file
    # Return a list of dicts, one per line
    pass


pretraining = load_jsonl(DATA_DIR / "pretraining_chunks.jsonl")
instructions = load_jsonl(DATA_DIR / "instruction_prompts.jsonl")
rlhf_pairs = load_jsonl(DATA_DIR / "rlhf_pairs.jsonl")

print(f"Pretraining chunks : {len(pretraining)}")
print(f"Instructions       : {len(instructions)}")
print(f"RLHF pairs         : {len(rlhf_pairs)}")

> ** Solution** — try the exercise above first, then click the **** button on the collapsed cell below to reveal and run the solution.


In [ ]:
def load_jsonl(path):
    records = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            records.append(json.loads(line))
    return records


pretraining = load_jsonl(DATA_DIR / "pretraining_chunks.jsonl")
instructions = load_jsonl(DATA_DIR / "instruction_prompts.jsonl")
rlhf_pairs = load_jsonl(DATA_DIR / "rlhf_pairs.jsonl")

print(f"Pretraining chunks : {len(pretraining)}")
print(f"Instruction prompts: {len(instructions)}")
print(f"RLHF pairs         : {len(rlhf_pairs)}")
print()
print("=== Pretraining chunk sample (first 200 chars) ===")
print(pretraining[0]["text"][:200])
print()
print("=== Instruction sample ===")
sample = instructions[0]
print(f"Instruction: {sample['instruction'][:100]}...")
print(f"Output: {sample['output'][:100]}...")

## Section 2: Tokenization Helper


In [ ]:
def tokenize_for_lm(examples, tokenizer, max_length=512):
    # TODO: tokenize examples["text"] with truncation and padding to max_length
    # Set labels = input_ids (causal LM: predict every token)
    pass

> ** Solution** — try the exercise above first, then click the **** button on the collapsed cell below to reveal and run the solution.


In [ ]:
def tokenize_for_lm(examples, tokenizer, max_length=512):
    encodings = tokenizer(
        examples["text"],
        truncation=True,
        max_length=max_length,
        padding="max_length",
        return_tensors=None,
    )
    encodings["labels"] = encodings["input_ids"].copy()
    return encodings

## Section 3: Continued Pretraining

**Continued pretraining** extends GPT-2's knowledge on raw novel text.  
The model learns the novel's vocabulary, style, and factual content at a statistical level.


In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# TODO: Load gpt2 model and tokenizer; set pad_token = eos_token
model_pt = None
tokenizer_pt = None

# TODO: Build HuggingFace Dataset from pretraining chunks
#       Map tokenize_for_lm over it (batched=True, remove "text" column)
#       Split 80 / 20 train / eval
train_pt = None
eval_pt = None

# TODO: Define TrainingArguments (2 epochs, batch=4, fp16=cuda_available)
# TODO: Create Trainer with DataCollatorForLanguageModeling(mlm=False)
# TODO: Call trainer.train()

> ** Solution** — try the exercise above first, then click the **** button on the collapsed cell below to reveal and run the solution.


In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

model_pt = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
tokenizer_pt = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer_pt.pad_token = tokenizer_pt.eos_token
print(f"Model parameters: {sum(p.numel() for p in model_pt.parameters()):,}")

pt_dataset = Dataset.from_list(pretraining)
pt_dataset = pt_dataset.map(
    lambda x: tokenize_for_lm(x, tokenizer_pt),
    batched=True,
    remove_columns=["text"],
)
pt_dataset.set_format("torch")
split = pt_dataset.train_test_split(test_size=0.2, seed=42)
train_pt, eval_pt = split["train"], split["test"]
print(f"Train: {len(train_pt)} | Eval: {len(eval_pt)}")

training_args_pt = TrainingArguments(
    output_dir=str(CKPT_DIR / "continued-pretraining"),
    num_train_epochs=2,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    warmup_steps=50,
    weight_decay=0.01,
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer_pt = Trainer(
    model=model_pt,
    args=training_args_pt,
    train_dataset=train_pt,
    eval_dataset=eval_pt,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer_pt, mlm=False),
)
print("Starting continued pretraining...")
trainer_pt.train()
print("Continued pretraining complete.")

## Section 4: Supervised Fine-Tuning (SFT)

Train the model to follow instructions by formatting data as `Instruction → Output` pairs.  
Loss is computed on the _output_ tokens only.


In [ ]:
from transformers import GPT2LMHeadModel

model_sft = None
tokenizer_sft = None


def format_instruction(item):
    # TODO: format as:
    #   ### Instruction:\n<instruction>\n\n### Response:\n<output>
    # (include ### Context: block only when item["input"] is non-empty)
    pass


# TODO: Load gpt2 model and tokenizer
# TODO: Build formatted dataset with format_instruction
# TODO: Tokenize with tokenize_for_lm, set format "torch", split 80/20
# TODO: TrainingArguments: 5 epochs, batch=2
# TODO: Train with Trainer + DataCollatorForLanguageModeling(mlm=False)

> ** Solution** — try the exercise above first, then click the **** button on the collapsed cell below to reveal and run the solution.


In [ ]:
from transformers import GPT2LMHeadModel

model_sft = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
tokenizer_sft = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer_sft.pad_token = tokenizer_sft.eos_token


def format_instruction(item):
    instruction = item["instruction"]
    input_text = item.get("input", "").strip()
    output = item["output"]
    if input_text:
        return {
            "text": f"### Instruction:\n{instruction}\n\n### Context:\n{input_text}\n\n### Response:\n{output}"
        }
    return {"text": f"### Instruction:\n{instruction}\n\n### Response:\n{output}"}


formatted = [format_instruction(i) for i in instructions]
sft_dataset = Dataset.from_list(formatted)
sft_tokenized = sft_dataset.map(
    lambda x: tokenize_for_lm(x, tokenizer_sft),
    batched=True,
    remove_columns=["text"],
)
sft_tokenized.set_format("torch")
split_sft = sft_tokenized.train_test_split(test_size=0.2, seed=42)
print(f"SFT train: {len(split_sft['train'])} | eval: {len(split_sft['test'])}")

training_args_sft = TrainingArguments(
    output_dir=str(CKPT_DIR / "sft"),
    num_train_epochs=5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    warmup_steps=10,
    weight_decay=0.01,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available(),
    report_to="none",
)
trainer_sft = Trainer(
    model=model_sft,
    args=training_args_sft,
    train_dataset=split_sft["train"],
    eval_dataset=split_sft["test"],
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer_sft, mlm=False),
)
print("Starting SFT training...")
trainer_sft.train()
print("SFT complete.")

## Section 5: Direct Preference Optimization (DPO)

Align the model to prefer "good" responses using paired preference data.  
Uses `trl.DPOTrainer` — requires the `trl` library.


In [ ]:
# TODO: Import DPOTrainer, DPOConfig from trl (guard with try/except)
# TODO: Load rlhf_pairs as a HuggingFace Dataset
# TODO: Split 85 / 15 train / eval
# TODO: Create DPOConfig with: 3 epochs, batch=2, lr=1e-5, beta=0.1, max_length=512
# TODO: Instantiate DPOTrainer with model, ref_model, dataset, processing_class=tokenizer
# TODO: Call trainer.train()

> ** Solution** — try the exercise above first, then click the **** button on the collapsed cell below to reveal and run the solution.


In [ ]:
try:
    from trl import DPOTrainer, DPOConfig

    TRL_AVAILABLE = True
    print("trl available — DPO enabled")
except ImportError:
    TRL_AVAILABLE = False
    print("trl not installed — run: pip install trl")

In [ ]:
if TRL_AVAILABLE:
    from transformers import GPT2LMHeadModel, GPT2Tokenizer

    model_dpo = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
    model_dpo_ref = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
    tokenizer_dpo = GPT2Tokenizer.from_pretrained("gpt2")
    tokenizer_dpo.pad_token = tokenizer_dpo.eos_token

    dpo_dataset = Dataset.from_list(rlhf_pairs)
    split_dpo = dpo_dataset.train_test_split(test_size=0.15, seed=42)
    print(f"DPO train: {len(split_dpo['train'])} | eval: {len(split_dpo['test'])}")

    dpo_config = DPOConfig(
        output_dir=str(CKPT_DIR / "dpo"),
        num_train_epochs=3,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        learning_rate=1e-5,
        beta=0.1,
        max_length=512,
        max_prompt_length=128,
        logging_steps=5,
        eval_strategy="epoch",
        save_strategy="epoch",
        fp16=torch.cuda.is_available(),
        report_to="none",
        remove_unused_columns=False,
    )

    trainer_dpo = DPOTrainer(
        model=model_dpo,
        ref_model=model_dpo_ref,
        args=dpo_config,
        train_dataset=split_dpo["train"],
        eval_dataset=split_dpo["test"],
        processing_class=tokenizer_dpo,
    )
    print("Starting DPO training...")
    trainer_dpo.train()
    print("DPO complete.")

## Section 6: LoRA Fine-Tuning

**Low-Rank Adaptation** injects small trainable matrices into frozen weights.  
Only ~0.5% of parameters are updated.  
Key hyperparameters: `r=16`, `lora_alpha=32`, `target_modules=["c_attn"]`.

![LoRA parameter efficiency: down-project through A, then up-project through B](images/lora-low-rank-adaptation.png)

For an input $x$, LoRA computes its update as $B(Ax)$, then adds it to the frozen layer's
usual output. In the diagram, $A \in \mathbb{R}^{4 \times 20}$ reduces a 20-value input to a
4-value bottleneck and $B \in \mathbb{R}^{10 \times 4}$ expands it into a 10-value adjustment.
The adapter therefore trains $80 + 40 = 120$ parameters, while the original 200 layer weights
stay frozen. A rank of 4 is intentionally large for this tiny example; transformer layers make
the same formula much more economical because $r$ is far smaller than the hidden dimension.


In [ ]:
from peft import get_peft_model, LoraConfig, TaskType

# TODO: Load gpt2 as base_model_lora and tokenizer_lora
# TODO: Create LoraConfig with:
#         task_type=TaskType.CAUSAL_LM, r=16, lora_alpha=32,
#         target_modules=["c_attn"], lora_dropout=0.05, bias="none"
# TODO: Apply with get_peft_model; call model.print_trainable_parameters()
# TODO: TrainingArguments: 3 epochs, batch=4, lr=3e-4
# TODO: Train with Trainer (reuse split_full from Section 5 / combined data)
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    # TODO: fill in remaining parameters
)

> ** Solution** — try the exercise above first, then click the **** button on the collapsed cell below to reveal and run the solution.


In [ ]:
try:
    from peft import get_peft_model, LoraConfig, TaskType

    PEFT_AVAILABLE = True
    print("peft available — LoRA enabled")
except ImportError:
    PEFT_AVAILABLE = False
    print("peft not installed — run: pip install peft")

In [ ]:
if PEFT_AVAILABLE:
    from transformers import GPT2LMHeadModel, GPT2Tokenizer
    from peft import get_peft_model, LoraConfig, TaskType

    # Build combined pretraining + instruction dataset for LoRA
    all_texts = [r["text"] for r in pretraining] + [
        f"### Instruction:\n{i['instruction']}\n\n### Response:\n{i['output']}"
        for i in instructions
    ]
    combined_ds = Dataset.from_dict({"text": all_texts})

    base_model_lora = GPT2LMHeadModel.from_pretrained("gpt2")
    tokenizer_lora = GPT2Tokenizer.from_pretrained("gpt2")
    tokenizer_lora.pad_token = tokenizer_lora.eos_token

    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=16,
        lora_alpha=32,
        target_modules=["c_attn"],
        lora_dropout=0.05,
        bias="none",
    )
    model_lora = get_peft_model(base_model_lora, lora_config).to(device)
    model_lora.print_trainable_parameters()

    lora_tokenized = combined_ds.map(
        lambda x: tokenize_for_lm(x, tokenizer_lora),
        batched=True,
        remove_columns=["text"],
    )
    lora_tokenized.set_format("torch")
    split_lora = lora_tokenized.train_test_split(test_size=0.1, seed=42)

    training_args_lora = TrainingArguments(
        output_dir=str(CKPT_DIR / "lora"),
        num_train_epochs=3,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        warmup_steps=50,
        weight_decay=0.01,
        learning_rate=3e-4,
        logging_steps=20,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        fp16=torch.cuda.is_available(),
        report_to="none",
    )
    trainer_lora = Trainer(
        model=model_lora,
        args=training_args_lora,
        train_dataset=split_lora["train"],
        eval_dataset=split_lora["test"],
        data_collator=DataCollatorForLanguageModeling(
            tokenizer=tokenizer_lora, mlm=False
        ),
    )
    print("Starting LoRA training...")
    trainer_lora.train()
    print("LoRA training complete.")

## Section 7: Evaluation

Test your trained models on questions of increasing difficulty about The Everglades Cipher.


In [ ]:
QUESTIONS = [
    {"level": "Easy", "q": "What is the name of Jake Malone's houseboat?"},
    {"level": "Easy", "q": "Where is Arturo Vasquez-Cortez's bookshop located?"},
    {"level": "Medium", "q": "What is the Codex Almeida and why is it significant?"},
    {"level": "Medium", "q": "How does Diego de Almeida's cipher system work?"},
    {
        "level": "Hard",
        "q": "What do the Philip IV letters reveal about the Templar dissolution?",
    },
    {
        "level": "Expert",
        "q": "Analyze the parallel between Diego de Almeida and Arturo Vasquez-Cortez.",
    },
]


def generate_text(model, tokenizer, prompt, max_new_tokens=100):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.8,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)


def evaluate_model(model, tokenizer, questions, max_new_tokens=120):
    # TODO: for each question in questions:
    #   format as: f"### Instruction:\n{item['q']}\n\n### Response:\n"
    #   call generate_text and extract the answer after "### Response:"
    #   return list of {"level", "question", "answer"} dicts
    pass


# TODO: run evaluate_model on your SFT model and print results

> ** Solution** — try the exercise above first, then click the **** button on the collapsed cell below to reveal and run the solution.


In [ ]:
def evaluate_model(model, tokenizer, questions, max_new_tokens=120):
    results = []
    for item in questions:
        prompt = f"### Instruction:\n{item['q']}\n\n### Response:\n"
        response = generate_text(model, tokenizer, prompt, max_new_tokens)
        if "### Response:" in response:
            answer = response.split("### Response:")[-1].strip()
        else:
            answer = response[len(prompt) :].strip()
        results.append(
            {
                "level": item["level"],
                "question": item["q"],
                "answer": answer[:300],
            }
        )
    return results


# Evaluate SFT model
print("=" * 60)
print("SFT MODEL RESPONSES")
print("=" * 60)
for r in evaluate_model(model_sft, tokenizer_sft, QUESTIONS):
    print(f"\n[{r['level']}] {r['question']}")
    print(f"  → {r['answer']}")

### Compare base model vs SFT


In [ ]:
# Compare base vs SFT on representative questions
from transformers import GPT2LMHeadModel, GPT2Tokenizer

base_model_eval = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
base_tok_eval = GPT2Tokenizer.from_pretrained("gpt2")
base_tok_eval.pad_token = base_tok_eval.eos_token

sample_qs = [QUESTIONS[0], QUESTIONS[2], QUESTIONS[4]]  # Easy, Medium, Hard

for item in sample_qs:
    prompt = f"### Instruction:\n{item['q']}\n\n### Response:\n"
    base_ans = generate_text(base_model_eval, base_tok_eval, prompt, 80)
    sft_ans = generate_text(model_sft, tokenizer_sft, prompt, 80)

    base_resp = base_ans.split("### Response:")[-1].strip()[:200]
    sft_resp = sft_ans.split("### Response:")[-1].strip()[:200]

    print(f"\n[{item['level']}] {item['q']}")
    print(f"  Input : {repr(prompt[:80])}...")
    print(f"  BASE  : {base_resp}")
    print(f"  SFT   : {sft_resp}")